A pedagogical walkthrough using the Kaggle Credit Card Fraud dataset (~0.17% positive rate).

**Plan**

1. Load and inspect the data.
2. Stratified train/val split.
3. Baseline: logistic regression on the full imbalanced training set.
4. Comparison: logistic regression on a 1:1 downsampled training set.
5. Compare metrics (Average Precision, ROC-AUC, precision/recall/F1 at threshold, confusion matrix).
6. Compare predicted-score distributions on the validation set.

## 1. Load data

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

# Registers and activates the shared `dvq` Plotly template.
import dvq_theme

# Load credentials from the repo-root .env, then map KAGGLE_API_TOKEN (KGAT_...) into KAGGLE_KEY for kagglehub.
load_dotenv(Path.cwd().parent.parent / ".env")
if os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_KEY"):
    os.environ["KAGGLE_KEY"] = os.environ["KAGGLE_API_TOKEN"]

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = Path(dataset_path) / "creditcard.csv"
print(csv_path)

df = pd.read_csv(csv_path)
print(df.shape)
df['Class'].value_counts(normalize=True)

/Users/dvq/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv


(284807, 31)


Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

## 2. Train/val split (stratified)

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train: {len(X_train):>7,}   positives: {int(y_train.sum()):>5,}  ({y_train.mean()*100:.3f}%)")
print(f"val:   {len(X_val):>7,}   positives: {int(y_val.sum()):>5,}  ({y_val.mean()*100:.3f}%)")

train: 227,845   positives:   394  (0.173%)
val:    56,962   positives:    98  (0.172%)


## 3. Baseline: full imbalanced training set

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)

scores_baseline = baseline.predict_proba(X_val)[:, 1]

## 4. Comparison: 1:1 downsampled training set

In [5]:
pos_idx = y_train[y_train == 1].index
neg_idx = y_train[y_train == 0].index
neg_sampled = np.random.RandomState(RANDOM_STATE).choice(neg_idx, size=len(pos_idx), replace=False)
ds_idx = np.concatenate([pos_idx, neg_sampled])

X_train_ds = X_train.loc[ds_idx]
y_train_ds = y_train.loc[ds_idx]
print(f"downsampled train: {len(X_train_ds):,}  (positives: {int(y_train_ds.sum()):,})")

downsampled = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
downsampled.fit(X_train_ds, y_train_ds)

scores_downsampled = downsampled.predict_proba(X_val)[:, 1]

downsampled train: 788  (positives: 394)


## 5. Metric comparison

In [6]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

def metrics_row(name, y_true, scores, threshold=0.5):
    y_pred = (scores >= threshold).astype(int)
    return {
        "model": name,
        "AP": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        f"precision@{threshold}": precision_score(y_true, y_pred, zero_division=0),
        f"recall@{threshold}": recall_score(y_true, y_pred, zero_division=0),
        f"f1@{threshold}": f1_score(y_true, y_pred, zero_division=0),
    }

metrics_df = pd.DataFrame([
    metrics_row("imbalanced (full)", y_val, scores_baseline),
    metrics_row("downsampled 1:1", y_val, scores_downsampled),
])
print(metrics_df.round(4).to_string(index=False))

            model     AP  ROC-AUC  precision@0.5  recall@0.5  f1@0.5
imbalanced (full) 0.7414   0.9605         0.8267      0.6327  0.7168
  downsampled 1:1 0.4578   0.9752         0.0516      0.9184  0.0977


In [7]:
for name, scores in [("imbalanced (full)", scores_baseline), ("downsampled 1:1", scores_downsampled)]:
    y_pred = (scores >= 0.5).astype(int)
    cm = confusion_matrix(y_val, y_pred)
    print(f"\n{name} @ threshold=0.5")
    print(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]))


imbalanced (full) @ threshold=0.5
          pred 0  pred 1
actual 0   56851      13
actual 1      36      62

downsampled 1:1 @ threshold=0.5
          pred 0  pred 1
actual 0   55209    1655
actual 1       8      90


Both metrics are built on ranking—they sweep thresholds from high to low—but they weight the ranking differently, which is why the downsampled model scores better on one and worse on the other.

**ROC-AUC** asks: for a random positive-negative pair, did the positive score higher? Averaged uniformly over all pairs.

**AP** steps down the ranked list from highest score to lowest, and each time it hits a positive, it notes the precision at that point. A positive found early (before negatives mix in) contributes more than one found late. So AP is specifically measuring the quality of high-confidence predictions—when the model is most certain something is fraud, is it actually right?

## 6. Score distributions on the validation set

Plotting each class separately with percentage on the y-axis keeps the two classes visually comparable, while hover tooltips surface the raw counts — so you can see both "what fraction of positives have a high score?" and "how many negatives is that tail actually?"

In [8]:
#| column: page
from IPython.display import HTML

bins = np.arange(0, 1.02, 0.02)
bin_centers = (bins[:-1] + bins[1:]) / 2
class_colors = {0: "#999999", 1: dvq_theme.ACCENT}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
    ),
    row_titles=["imbalanced (full)", "downsampled 1:1"],
    shared_yaxes="all",
    shared_xaxes="all",
    vertical_spacing=0.14,
    horizontal_spacing=0.1,
)

for row, (name, scores) in enumerate(
    [("imbalanced (full)", scores_baseline), ("downsampled 1:1", scores_downsampled)],
    start=1,
):
    for col, label in enumerate([0, 1], start=1):
        mask = y_val == label
        counts, _ = np.histogram(scores[mask], bins=bins)
        pct = counts / counts.sum() * 100
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=pct,
                width=0.02,
                customdata=counts,
                hovertemplate="score: %{x:.2f}<br>%{y:.1f}%<br>count: %{customdata:,d}<extra></extra>",
                marker=dict(color=class_colors[label], line=dict(width=0)),
                showlegend=False,
            ),
            row=row, col=col,
        )

        n_above = int((scores[mask] >= 0.5).sum())
        fig.add_vline(
            x=0.5, row=row, col=col,
            line_dash="dash", line_color="#444444", line_width=1.5,
        )
        fig.add_annotation(
            x=0.52, y=88,
            text=f"{n_above:,} above 0.5",
            row=row, col=col,
            showarrow=False,
            xanchor="left",
            font=dict(size=11, color="#444444"),
        )

fig.update_layout(
    height=560,
    autosize=True,
    title="Score distributions by class — % within class (hover for raw count)",
    bargap=0,
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)
fig.update_xaxes(title_text="predicted P(class=1)", range=[0, 1])
fig.update_yaxes(title_text="% of class", range=[0, 100])

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-scores-2x2",
    config={"responsive": True},
    default_width="100%",
    default_height="560px",
))

Now the picture is hard to miss on the positive side (right column): the downsampled model pushes ~80% of all fraud cases into the last bin, while the baseline spreads them across the full range. The negative side (left column) tells a subtler story. You won't see any bars above 0.5 in the bottom-left — hover over that region and you'll find they're there, just under 0.2% per bin. That's because 1,655 out of 56,864 negatives is only 2.9%, and spread across 25 bins it's too small to appear on a linear scale.

But here's the thing: those invisible 1,655 negatives are competing directly with only 90 true positives in the high-score region. Flag everything above 0.5 and you send out 1,745 alerts, 90 of which are real fraud — a 5% precision. For every genuine fraud case, an analyst wades through roughly 18 false alarms.

The baseline avoids this because its negative tail is almost nonexistent above 0.2 (top-left). Fewer negatives leak into the high-score region, so precision at the same threshold is 83%.

## Why does training on downsampled data push scores higher?

Logistic regression models the probability of a positive outcome as:

$$p = \text{sigmoid}(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

The weights $w$ learn which features push the probability up or down. The bias $b$ is a constant added to every sample's raw score before it gets squashed through sigmoid — the model's baseline suspicion before it looks at any features.

The model is trained to minimize cross-entropy loss:

$$L = -\sum_i \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

The $\log$ terms are what give this its character: $\log(p)$ shoots toward $-\infty$ as $p \to 0$, so being confidently wrong incurs a catastrophically large penalty. Think of it as a **"how surprised were you?" scorer**. Every time the model makes a prediction, it gets scored on how shocked it should be by what actually happened. Predict 99% fraud and it turns out to be fraud — barely surprised, tiny penalty. Predict 1% fraud and it turns out to be fraud — extremely surprised, huge penalty. The model's entire job during training is to stop being surprised.

Now zoom in on the bias. If it's set too low, the model is systematically shocked every time fraud shows up. If it's too high, it's systematically shocked by the 99.83% of transactions that aren't fraud. The only resting point where the bias stops accumulating surprise from both sides is when it matches how often fraud actually occurs in training — the training positive rate. Any other value leaves the model consistently surprised by one class or the other, which keeps bleeding into the loss.

This is why the downsampled model's scores are inflated. It was trained to stop being surprised in a world where fraud is 50% common. When deployed into a world where fraud is 0.17% common, its baseline suspicion is miscalibrated by three orders of magnitude. The weights still learned the right patterns — the model reliably ranks fraud higher than non-fraud, which is why ROC-AUC is actually better — but the absolute score values reflect the wrong world. That's what miscalibration means: the scores don't correspond to real-world frequencies.

## What happens when we use `class_weight='balanced'`?